# Heart Disease Prediction — Full Analysis

This notebook is the single entry point for the full project workflow. Run all cells **top-to-bottom** and everything will execute: data loading, EDA, model training, evaluation, XGBoost, SHAP explainability, and the Streamlit web app.

**ProjectEC5.1** fixes the following issues from EC5:
- Section numbering corrected — sections now run 1 through 13 sequentially
- CI/CD updated: `tests.yml` upgraded to Node.js 24 + `pr_checks.yml` added (flake8, bandit, mypy)
- Test count corrected — 66 tests (restored from EC4 regression)
- Summary table updated to reflect all EC5 features
- Ethical reflection updated to mention SHAP and XGBoost

**Before opening this notebook**, make sure you have set up the virtual environment and launched Jupyter from inside it.

### First-time setup

```bash
# From the project root (ProjectEC5/)

# Windows — create venv with Python 3.11
py -3.11 -m venv .venv

# Activate (Windows)
.venv\Scripts\Activate.ps1

# macOS / Linux — uncomment the two lines below instead:
# python3.11 -m venv .venv
# source .venv/bin/activate

# Install dependencies (Windows path — use explicit venv python to avoid global Python bleeding in)
.venv/Scripts/python.exe -m pip install --upgrade pip
.venv/Scripts/python.exe -m pip install -r requirements.txt
.venv/Scripts/python.exe -m pip install ipykernel jupyter
.venv/Scripts/python.exe -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"

# macOS / Linux — uncomment the four lines below instead:
# python3 -m pip install --upgrade pip
# python3 -m pip install -r requirements.txt
# python3 -m pip install ipykernel jupyter
# python3 -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"
```

### Verify the venv is correct

```bash
# Windows:
.venv/Scripts/python.exe -m pip show ipykernel jupyter-core
# Both should show Location: <project-root>\.venv\...

# macOS / Linux:
# python3 -m pip show ipykernel jupyter-core
# Both should show Location: <project-root>/.venv/...
```

### Select the kernel in VS Code

1. `Ctrl+Shift+P` → **Developer: Reload Window**
2. Open the notebook
3. Kernel selector (top-right) → **Select Another Kernel** → **Jupyter Kernel**
4. Select **"ProjectEC5 venv"** or **"Python 3 (ipykernel)"**

### If the kernel still won't start — force reinstall ipykernel

```bash
# Windows:
.venv/Scripts/python.exe -m pip install -U ipykernel
.venv/Scripts/python.exe -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"

# macOS / Linux:
# python3 -m pip install -U ipykernel
# python3 -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"
```

Then fully close and reopen VS Code.

### If you need to recreate the venv from scratch

```bash
# Windows:
deactivate
rm -rf .venv
py -3.11 -m venv .venv
.venv\Scripts\Activate.ps1
.venv/Scripts/python.exe -m pip install --upgrade pip
.venv/Scripts/python.exe -m pip install -r requirements.txt
.venv/Scripts/python.exe -m pip install ipykernel jupyter
.venv/Scripts/python.exe -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"

# macOS / Linux — uncomment the lines below instead:
# deactivate
# rm -rf .venv
# python3.11 -m venv .venv
# source .venv/bin/activate
# python3 -m pip install --upgrade pip
# python3 -m pip install -r requirements.txt
# python3 -m pip install ipykernel jupyter
# python3 -m ipykernel install --sys-prefix --display-name "ProjectEC5 venv"
```

> **Why use the explicit venv python path instead of just `pip`?**
> With multiple Python versions installed, plain `pip` can install packages into the wrong
> global Python. Using the explicit venv path guarantees everything lands in the venv.
> On macOS/Linux the activated venv handles this automatically.

Once the kernel is selected, click **Run All**.

**Contents:**
1. Environment check
2. Data preparation
3. Imports & path setup
4. Data loading and cleaning
5. Exploratory Data Analysis (EDA)
6. Model training
7. Evaluation & visualisations
8. Hyperparameter tuning
9. XGBoost — 4th model
10. SHAP — explainability
11. Streamlit app — launch & inline results
12. Ethical reflection
13. Terminal app
14. CI/CD — GitHub Actions
15. Summary


---
## 1. Environment Check

This cell verifies that the notebook is running inside the project's `.venv` virtual environment and that the Python version is compatible. If the path shown does not contain `.venv`, stop here and relaunch Jupyter from the activated environment.


In [ ]:
import sys

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")

if ".venv" not in sys.executable and "venv" not in sys.executable:
    print("\n⚠️  WARNING: This kernel does not appear to be running inside a virtual environment.")
    print("   Activate .venv and relaunch Jupyter to ensure the correct packages are used.")
else:
    print("\n✅ Virtual environment detected — you're good to go.")
# Please run the foloowing in Git Bash if virtual env is not present:
# Close the notebook, then create the venv first:
# cd <your-project-root>
# py -3.11 -m venv .venv
# source .venv/Scripts/activate
# pip install -r requirements.txt
# pip install ipykernel
# python -m ipykernel install --user --name=ProjectEC5 --display-name "Python (ProjectEC5)"

In [ ]:
# Verify all required packages are installed.
# If anything is missing, installs from requirements.txt automatically.
import importlib
import subprocess

required = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn", "joblib", "streamlit"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("Installing from requirements.txt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(Path(PROJECT_ROOT) / "requirements.txt")])
    print("✅ Done — restart the kernel and run again.")
else:
    print("✅ All required packages found.")

---
## 2. Data Preparation

Before loading data we run `src/data_preparation.py` to prepare the Cleveland Heart Disease dataset.

This step:
- Loads `processed.cleveland.data` (303 rows, 14 columns)
- Converts target to binary (0 = no disease, 1+ = disease present)
- Imputes 6 missing values in `ca` and `thal` with column medians
- Saves to `data/heart_combined.csv`

The Cleveland dataset is used because it is the only UCI hospital dataset with reliable
measurements for all 13 features including `ca` and `thal` — the two most important
predictive features (~28% combined importance). See the Investigation Findings section
in README.md for the full analysis.

The original file is never modified. Run this cell once — subsequent runs are safe.

In [ ]:
# Run data preparation pipeline
# Loads processed.cleveland.data, imputes 6 missing values,
# converts target to binary and saves to heart_combined.csv
import subprocess
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath('__file__')).parent.parent

result = subprocess.run(
    [sys.executable, '-m', 'src.data_preparation'],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT)  # run from project root so src is found as a module
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('✅ heart_combined.csv ready')


---
## 3. Imports & Path Setup

We resolve `PROJECT_ROOT` relative to this notebook's own location so imports work correctly regardless of where Jupyter was launched from. Then we import all libraries and configure paths used throughout the notebook.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

# Resolve project root from this notebook's location (notebooks/ -> ..)
# Works correctly regardless of the directory Jupyter was launched from.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Notebook dir      : {NOTEBOOK_DIR}")

In [ ]:
import json
import socket
import subprocess
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, IFrame, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

from src.data_processing import DataProcessor
from src.model_training import ModelTrainer

# Plotting defaults
plt.style.use("dark_background")
sns.set_theme(style="darkgrid", palette="bright")
plt.rcParams["figure.dpi"] = 110

# Central path constants — all file I/O uses these
DATA_PATH    = Path(PROJECT_ROOT) / "data" / "heart_combined.csv"
MODELS_DIR   = Path(PROJECT_ROOT) / "models"
MODEL_PATH   = MODELS_DIR / "random_forest.pkl"
RESULTS_PATH = MODELS_DIR / "training_results.json"

print("✅ All imports OK")
print(f"Dataset path : {DATA_PATH}  (exists: {DATA_PATH.exists()})")

---
## 4. Data Loading and Cleaning

We use `DataProcessor` from `src/data_processing.py` to load the UCI Heart Disease CSV and apply the project's standard cleaning steps: column name normalisation and missing value handling. After loading, we inspect the shape, the first few rows, and the class balance of the target variable.

In [ ]:
# Instantiate the DataProcessor and run load + clean
processor = DataProcessor(data_path=str(DATA_PATH))
processor.load_data()
processor.clean_data()

df = processor.df
print(f"Dataset shape  : {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)")
print(f"Columns        : {df.columns.tolist()}")
df.head()

In [ ]:
# Built-in summary from DataProcessor: descriptive statistics for all columns
print("=== Dataset Summary ===")
processor.summary()

In [ ]:
# Data quality check: missing values and class balance
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTarget distribution (0 = no disease, 1 = disease):")
print(df["target"].value_counts())
print(f"\nClass balance: {df['target'].mean():.1%} positive (heart disease present)")

---
## 5. Exploratory Data Analysis (EDA)

Before training we explore the data visually to understand distributions, class separation, and relationships between features. This helps motivate feature selection and model choices.

We produce four sets of charts:
- **Target distribution** and **age histogram** split by target class
- **Numeric feature distributions** split by target
- **Categorical feature proportions** vs target
- **Correlation heatmap** across all features

In [ ]:
# --- Target distribution and age histogram side by side ---
# The left chart shows overall class balance (important: a heavily imbalanced
# dataset would require resampling). The right shows whether age alone
# separates the two classes.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df["target"].value_counts().sort_index()
axes[0].bar(["No Disease (0)", "Heart Disease (1)"], target_counts.values,
            color=["steelblue", "tomato"], edgecolor="white")
axes[0].set_title("Target Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

for label, color in [(0, "steelblue"), (1, "tomato")]:
    axes[1].hist(df[df["target"] == label]["age"], bins=20, alpha=0.6,
                 color=color, label=f"Target={label}")
axes[1].set_title("Age Distribution by Target")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Numeric feature distributions split by target class ---
# Overlapping histograms reveal which continuous features discriminate well.
# For example, thalach (max heart rate) tends to be higher in disease-positive patients.
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 4))

for ax, col in zip(axes, numeric_features):
    for label, color in [(0, "steelblue"), (1, "tomato")]:
        ax.hist(df[df["target"] == label][col], bins=20, alpha=0.6,
                color=color, label=f"Target={label}")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend(fontsize=7)

fig.suptitle("Key Feature Distributions by Target", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Categorical features: proportion of disease-positive cases per category ---
# Each bar shows what fraction of patients in that category have heart disease.
# Chest pain type (cp) and exercise-induced angina (exang) show strong separation.
cat_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, col in zip(axes.flatten(), cat_features):
    ct = pd.crosstab(df[col], df["target"], normalize="index")
    ct.plot(kind="bar", ax=ax, color=["steelblue", "tomato"],
            edgecolor="white", legend=False)
    ax.set_title(f"{col} vs target")
    ax.set_xlabel(col)
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=0)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["steelblue", "tomato"]]
fig.legend(handles, ["No Disease", "Heart Disease"], loc="lower right", ncol=2)
plt.suptitle("Categorical Features vs Target", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
# Shows linear correlations between all features. Strong correlations with
# `target` suggest predictive value. High inter-feature correlations could
# cause multicollinearity issues for linear models.
fig, ax = plt.subplots(figsize=(12, 9))
corr = processor.correlation_matrix()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (redundant)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Model Training

We split the data into training (80%) and test (20%) sets using stratified sampling so both splits have the same class balance. Then `ModelTrainer` from `src/model_training.py` builds and trains three scikit-learn pipelines:

- **Logistic Regression** — a simple, interpretable linear baseline
- **Random Forest** — an ensemble of decision trees that captures non-linear relationships
- **Decision Tree** — a single tree included to show the value of ensembling: comparing it against Random Forest demonstrates how much accuracy is gained by combining many trees instead of one

After training, all model files and evaluation metrics are saved to `models/` so the Streamlit app can load them without re-training.

In [ ]:
# Split into features (X) and target (y), then train/test split
X, y = processor.get_features_and_target()
X_train, X_test, y_train, y_test = processor.split_data(test_size=0.2, random_state=42)

print(f"Train size : {len(X_train)} samples  |  Test size : {len(X_test)} samples")
print(f"Train positive rate : {y_train.mean():.1%}")
print(f"Test  positive rate : {y_test.mean():.1%}  (stratification check)")

In [ ]:
# Build pipelines, train all three models, and evaluate on the held-out test set
trainer = ModelTrainer(random_state=42)
trainer.build_pipelines()
trainer.train_models(X_train, y_train)
results = trainer.evaluate(X_test, y_test)

print("\n=== Model Comparison ===")
print(trainer.compare().to_string())

In [ ]:
# Save all artefacts needed by the Streamlit app:
#   models/logistic_regression.pkl  — loaded by app.py
#   models/random_forest.pkl        — loaded by app.py
#   models/training_results.json    — read by the Model Performance page

MODELS_DIR.mkdir(exist_ok=True)

# Save best model via ModelTrainer's built-in method
trainer.save_best_model(str(MODEL_PATH))

# Save individual .pkl files (app.py looks for these by model slug)
for name, result in results.items():
    slug = name.lower().replace(" ", "_")
    joblib.dump(result.pipeline, MODELS_DIR / f"{slug}.pkl")

# Build and save training_results.json
results_dict = {
    name: {
        "accuracy":  round(r.accuracy, 4),
        "f1":        round(r.f1, 4),
        "precision": round(r.precision, 4),
        "recall":    round(r.recall, 4),
        "roc_auc":   round(r.roc_auc, 4),
    }
    for name, r in results.items()
}
with RESULTS_PATH.open("w") as f:
    json.dump(results_dict, f, indent=2)

print(f"✅ Models saved  : {MODELS_DIR}")
print(f"✅ Results saved : {RESULTS_PATH}")

### Model Results — Evolution from EC3 to EC5

#### What changed across versions

| Version | Goal | What was found | Outcome |
|---------|------|----------------|---------|
| **EC3** | Build baseline pipeline — 3 models, terminal app, Streamlit | Working pipeline on heart.csv (1025 rows) | RF 0.885 — **later found to be inflated** |
| **EC4** | Add Kaggle dataset as new data source to increase training data | 723 duplicate rows discovered during combination | Duplicates removed → RF 0.902 honest result ✅ |
| **EC4** | Combine Cleveland + Kaggle → 1220 rows | ca + thal absent from Kaggle → imputed or dropped | **More data did not improve accuracy** ⚠️ |
| **EC4** | Find best dataset strategy via systematic investigation | Cleveland only with all 13 features wins | Decision justified by evidence ✅ |
| **EC4** | Squeeze more from the best dataset | Hyperparameter tuning (GridSearchCV, cv=5) | Decision Tree +8.2% — biggest winner |
| **EC5** | Add XGBoost as 4th model | Matches LR baseline without any tuning | Confirms dataset quality is solid |
| **EC5** | Address black box criticism from ethics section | SHAP explainability | thal, ca, cp confirmed as top-3 features |

#### EC4 Key Finding — Adding a New Data Source Did Not Improve Accuracy

EC4's goal was to add the Kaggle Heart Failure Prediction dataset (fedesoriano, 2021) as a
new data source — combining five hospital datasets for 1220 rows vs the original 302.

During the combination process, **723 duplicate rows** were found in the original `heart.csv`.
This explained the inflated EC3 results — the same patients appeared in both train and test sets.

A systematic investigation (`scripts/investigate_datasets.py`) then tested every combination:

| Approach | Rows | Features | RF Accuracy | Why it failed |
|----------|------|----------|-------------|---------------|
| EC3 original (with duplicates) | 1025 | 13 | 1.000 | ❌ Overfitting — data leakage from duplicates |
| EC3 deduplicated | 302 | 13 | ~0.885 | Honest baseline |
| UCI 4 hospitals — drop ca + thal | 918 | 11 | ~0.672 | ❌ Feature loss too costly |
| Kaggle combined — impute ca + thal | 1220 | 13 | ~0.643 | ❌ Imputing 90%+ of rows is unreliable |
| **Cleveland only — all 13 features** | **303** | **13** | **0.902** | ✅ Selected |

**Why more data hurt:** `ca` and `thal` are the two most important features (~28% combined
importance). The Kaggle dataset and other UCI hospitals do not have reliable measurements for
these features — they are missing in 90%+ of rows. Adding those rows requires either dropping
or imputing ca + thal — both options lose more predictive power than the extra rows provide.
303 high-quality rows outperform 918–1220 rows with compromised features.

This is the core engineering insight of EC4: **data quality beats data quantity**.

#### EC3 — baseline results (1025 rows, with duplicates)

| Model | Accuracy | ROC AUC | Note |
|-------|----------|---------|------|
| Random Forest | 0.885 | 0.955 | ⚠️ Inflated — 723 duplicate rows |
| Logistic Regression | 0.869 | 0.935 | ⚠️ Inflated |
| Decision Tree | 0.754 | 0.754 | ⚠️ Inflated |

#### EC4 — after deduplication + hyperparameter tuning (303 rows, 13 features)

| Model | Default Acc | Tuned Acc | Default AUC | Tuned AUC |
|-------|------------|-----------|-------------|-----------|
| Random Forest | 0.902 | 0.902 | 0.955 | 0.958 |
| Logistic Regression | 0.869 | 0.853 | 0.951 | 0.958 |
| Decision Tree | 0.787 | **0.869** | 0.808 | **0.871** |

Decision Tree was the biggest winner from tuning (+8.2% accuracy).

#### EC5 — with XGBoost (303 rows, 13 features, actual results)

| Model | Accuracy | F1 | Precision | Recall | ROC AUC |
|-------|----------|----|-----------|--------|---------|
| Random Forest | 0.9016 | 0.9000 | 0.8438 | **0.9643** | 0.9545 |
| Logistic Regression | 0.8689 | 0.8667 | 0.8125 | 0.9286 | 0.9513 |
| XGBoost | 0.8689 | 0.8667 | 0.8125 | 0.9286 | 0.9102 |
| Decision Tree | 0.7869 | 0.7937 | 0.7143 | 0.8929 | 0.8084 |
| Decision Tree (tuned) | **0.869** | — | — | — | **0.871** |

Random Forest is the best model on all key metrics. In a medical context **Recall** is the most
critical metric — a missed disease case is more costly than a false alarm.
Random Forest leads on recall (96.4%).

XGBoost matches Logistic Regression exactly on accuracy and F1 without any tuning —
confirming its strength for tabular data and validating the dataset quality.


---
## 7. Evaluation & Visualisations

We evaluate both models on the test set using five metrics:

- **Accuracy** — overall fraction of correct predictions
- **Precision** — of all predicted positives, how many are truly positive
- **Recall** — of all actual positives, how many did the model catch
- **F1-score** — harmonic mean of precision and recall
- **ROC AUC** — area under the ROC curve; measures ranking quality across all thresholds

In a medical context recall is especially important: missing a true disease case (false negative) is more costly than a false alarm.

In [ ]:
# --- Side-by-side bar chart of all five metrics for both models ---
metrics = ["accuracy", "f1", "precision", "recall", "roc_auc"]
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate(zip(results_dict.keys(), ["steelblue", "tomato", "seagreen"])):
    vals = [results_dict[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace("_", " ").title() for m in metrics])
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — All Metrics")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Confusion matrices ---
# Each cell shows: top-left = true negatives, top-right = false positives,
# bottom-left = false negatives (missed disease), bottom-right = true positives.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result.pipeline.predict(X_test))
    disp = ConfusionMatrixDisplay(cm, display_labels=["No Disease", "Heart Disease"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC curves ---
# The ROC curve plots true positive rate vs false positive rate at every
# decision threshold. A higher AUC means better discrimination between classes.
fig, ax = plt.subplots(figsize=(8, 6))

for (name, result), color in zip(results.items(), ["steelblue", "tomato", "seagreen"]):
    if hasattr(result.pipeline, "predict_proba"):
        y_score = result.pipeline.predict_proba(X_test)[:, 1]
    else:
        y_score = result.pipeline.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest feature importances ---
# Feature importance shows which input variables the forest relies on most.
# This helps explain the model and highlights clinically meaningful predictors.
rf_result = results.get("Random Forest")
if rf_result is not None:
    pipeline = rf_result.pipeline
    # Try common step names; fall back to the last step in the pipeline
    clf = (
        pipeline.named_steps.get("classifier")
        or pipeline.named_steps.get("model")
        or list(pipeline.named_steps.values())[-1]
    )
    if hasattr(clf, "feature_importances_"):
        feature_names = X.columns.tolist()
        importances = clf.feature_importances_
        indices = np.argsort(importances)[::-1]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(range(len(feature_names)),
               importances[indices], color="steelblue", edgecolor="white")
        ax.set_xticks(range(len(feature_names)))
        ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha="right")
        ax.set_title("Random Forest — Feature Importances")
        ax.set_ylabel("Importance (mean decrease in impurity)")
        plt.tight_layout()
        plt.show()
    else:
        print("Feature importances not available for this pipeline configuration.")
else:
    print("Random Forest result not found in results dict.")

In [ ]:
# --- Full classification reports ---
# Per-class precision, recall, and F1. Useful for checking if one class
# is being systematically under-predicted.
for name, result in results.items():
    print(f"\n{'='*52}")
    print(f" Classification Report — {name}")
    print(f"{'='*52}")
    print(classification_report(
        y_test,
        result.pipeline.predict(X_test),
        target_names=["No Disease", "Heart Disease"]
    ))

---
## 8. Hyperparameter Tuning

We use `GridSearchCV` with 5-fold cross-validation to find the best hyperparameters
for each model. This is more reliable than a single train/test split since it
evaluates performance across multiple data splits.

Scoring metric: **ROC AUC** — chosen because it measures ranking quality across
all decision thresholds, making it robust for imbalanced classes.

Parameters tuned:
- **Logistic Regression**: regularisation strength (`C`), solver
- **Random Forest**: number of trees, max depth, min samples split, max features
- **Decision Tree**: max depth, min samples split, min samples leaf, criterion

Note: tuning on 303 rows with 5-fold CV uses ~242 rows for training each fold.
Results may vary slightly between runs.

In [ ]:
# Hyperparameter tuning using GridSearchCV with 5-fold cross-validation
# This may take 1-2 minutes on 303 rows
print("Tuning models — this may take a moment...\n")

tuned_trainer = ModelTrainer(random_state=42)
tuned_trainer.tune_models(X_train, y_train, cv=5)
tuned_results = tuned_trainer.evaluate(X_test, y_test)

print("\n=== Tuned Model Comparison ===")
print(tuned_trainer.compare().to_string())

In [ ]:
# --- Compare default vs tuned results ---
default_df = pd.DataFrame(results_dict).T.round(4)
default_df.index.name = 'Model'
default_df.columns = [c.replace('_', ' ').title() for c in default_df.columns]

tuned_dict = {
    name: {
        'accuracy':  round(r.accuracy, 4),
        'f1':        round(r.f1, 4),
        'precision': round(r.precision, 4),
        'recall':    round(r.recall, 4),
        'roc_auc':   round(r.roc_auc, 4),
    }
    for name, r in tuned_results.items()
}
tuned_df = pd.DataFrame(tuned_dict).T.round(4)
tuned_df.index.name = 'Model'
tuned_df.columns = [c.replace('_', ' ').title() for c in tuned_df.columns]

print('=== Default Parameters ===')
print(default_df.to_string())
print()
print('=== Tuned Parameters ===')
print(tuned_df.to_string())

# Side-by-side accuracy comparison
fig, ax = plt.subplots(figsize=(10, 5))
models = list(results_dict.keys())
x = np.arange(len(models))
width = 0.35
default_acc = [results_dict[m]['accuracy'] for m in models]
tuned_acc = [tuned_dict[m]['accuracy'] for m in models]

bars1 = ax.bar(x - width/2, default_acc, width, label='Default', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, tuned_acc, width, label='Tuned', color='tomato', alpha=0.85)

for bar, v in zip(bars1, default_acc):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
for bar, v in zip(bars2, tuned_acc):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Accuracy')
ax.set_title('Default vs Tuned Model Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. XGBoost — 4th Model (ProjectEC5)

ProjectEC5 adds **XGBoost (Extreme Gradient Boosting)** as a fourth model.
XGBoost builds trees sequentially — each new tree corrects the errors of the previous one.
It is the industry standard for tabular data and often outperforms Random Forest with less tuning.

The model is trained with default parameters and compared against all three baseline models.


In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# --- Train and evaluate XGBoost with default parameters ---
xgb_pipeline = Pipeline([
    ('model', XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ))
])
xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)
xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_metrics = {
    'accuracy':  round(accuracy_score(y_test, xgb_pred), 4),
    'f1':        round(f1_score(y_test, xgb_pred), 4),
    'precision': round(precision_score(y_test, xgb_pred), 4),
    'recall':    round(recall_score(y_test, xgb_pred), 4),
    'roc_auc':   round(roc_auc_score(y_test, xgb_prob), 4),
}

print('XGBoost (default) metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k:<12}: {v}')

In [ ]:
# --- Compare all 4 models ---
all_models = {
    'Logistic Regression': results_dict['Logistic Regression'],
    'Random Forest':       results_dict['Random Forest'],
    'Decision Tree':       results_dict['Decision Tree'],
    'XGBoost':             xgb_metrics,
}

all_df = pd.DataFrame(all_models).T.round(4)
all_df.index.name = 'Model'
all_df.columns = [c.replace('_', ' ').title() for c in all_df.columns]
all_df = all_df.sort_values('Accuracy', ascending=False)

print('All 4 models — EC5 comparison:')
print(all_df.to_string())

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
all_df['Accuracy'].plot(kind='bar', ax=axes[0], color='steelblue', title='Accuracy')
all_df['Roc Auc'].plot(kind='bar', ax=axes[1], color='darkorange', title='ROC AUC')
for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

---
## 10. SHAP — Explainability (ProjectEC5)

**SHAP (SHapley Additive exPlanations)** explains *why* a model made a specific prediction.
It comes from game theory and fairly distributes the contribution of each feature to the prediction.

This directly addresses the *black box* criticism in the ethics section — a doctor needs to understand
why a model predicts high risk before acting on it.

We produce three plots:
- **Beeswarm (Random Forest)** — all patients, all features. Global importance and direction.
- **Waterfall (Random Forest)** — one high-risk patient explained step by step.
- **Beeswarm (XGBoost)** — compare feature weighting between models.


In [ ]:
import shap

# --- SHAP beeswarm plot — Random Forest (all test patients) ---
rf_model = trainer.results['Random Forest'].pipeline.named_steps['model']
explainer_rf = shap.TreeExplainer(rf_model)
shap_values_rf = explainer_rf(X_test)

# For binary classification take class 1 (disease present)
sv = shap_values_rf[:, :, 1] if len(shap_values_rf.shape) == 3 else shap_values_rf

print('SHAP Beeswarm Plot — Random Forest')
print('Each dot = one patient. Red = high feature value, Blue = low feature value.')
print('Position on x-axis = SHAP value (impact on prediction)\n')
shap.plots.beeswarm(sv, max_display=13, show=True)

### 🔍 Beeswarm Interpretation — Random Forest

The three most influential features are **thal**, **ca**, and **cp** — their dots spread widest across the x-axis, meaning they have the greatest impact on predictions.

| Feature | What it means |
|---|---|
| **thal** | Low thal values strongly push toward a heart disease prediction. The most impactful feature overall. |
| **ca** | Fewer major vessels (low value) increases predicted risk significantly. |
| **cp** | Certain chest pain types (high value, red) are strong indicators of disease. |
| **oldpeak** | High ST depression values spread both directions — complex relationship with risk. |
| **thalach** | Low maximum heart rate (blue) is associated with higher predicted risk. |
| **exang** | Exercise-induced angina (yes) pushes predictions toward disease. |
| **sex** | Male patients tend toward higher predicted risk in this model. |
| **age, chol, trestbps** | Relatively low impact — dots clustered near zero. |
| **fbs, restecg** | Minimal influence on predictions in this model. |

> **Note:** Features at the top are ranked by importance — `thal`, `ca`, and `cp` should be prioritised in clinical screening.

In [ ]:
# --- SHAP waterfall plot — one patient explained ---
# Pick the first high-risk patient from test set
high_risk_idx = y_test[y_test == 1].index[0]
patient_pos = X_test.index.get_loc(high_risk_idx)

print(f'Explaining prediction for patient index {high_risk_idx} (high risk)')
print(f'Actual label: {y_test[high_risk_idx]}, ',
      f'Predicted: {trainer.results["Random Forest"].pipeline.predict(X_test.iloc[[patient_pos]])[0]}')
print()
shap.plots.waterfall(sv[patient_pos], max_display=13, show=True)

### 🔍 Waterfall Interpretation — Single High-Risk Patient

This plot explains **one individual prediction** — the first high-risk patient in the test set.

- Each bar shows how much a single feature **pushed the prediction up (red)** or **down (blue)**
- **f(x)** = the final model output for this patient
- **E[f(x)]** = the baseline (average prediction across all patients)

The waterfall shows the exact reasoning chain the model used — making the prediction auditable and explainable to a clinician.

In [ ]:
# --- SHAP beeswarm — XGBoost ---
explainer_xgb = shap.TreeExplainer(xgb_pipeline.named_steps['model'])
shap_values_xgb = explainer_xgb(X_test)

sv_xgb = shap_values_xgb[:, :, 1] if len(shap_values_xgb.shape) == 3 else shap_values_xgb

print('SHAP Beeswarm Plot — XGBoost')
shap.plots.beeswarm(sv_xgb, max_display=13, show=True)

### 🔍 Beeswarm Interpretation — XGBoost

Compared to Random Forest, XGBoost shows a similar feature ranking with **thal**, **ca**, and **cp** again dominating. Key differences to note:

- XGBoost tends to produce sharper, more concentrated SHAP values
- If the spread differs significantly from Random Forest, it suggests the two models are weighting features differently — worth noting in the ethical reflection

---
## 11. Streamlit App — Launch & Inline Results

The Streamlit app (`app.py`) provides a browser-based interface with four pages: **Home**, **Prediction**, **Model Performance**, and **About**.

The cell below starts the app as a background process (using the same Python interpreter as this notebook, so it shares the `.venv`). It then waits up to 20 seconds for the server to become ready before embedding it inline via an `IFrame`.

The subsequent cells reproduce the **Model Performance** page charts directly in the notebook using the `training_results.json` saved during training — so the results are visible here even if the iframe does not render in your environment (e.g. VS Code).

**To open the app manually instead**, run in a terminal:
```bash
py -m src.main --streamlit
```


In [ ]:
STREAMLIT_PORT = 8501

def is_port_open(port, host="localhost"):
    """Return True if a process is already listening on the port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0

if is_port_open(STREAMLIT_PORT):
    print(f"Streamlit is already running on port {STREAMLIT_PORT}.")
else:
    app_path = str(Path(PROJECT_ROOT) / "app.py")
    _proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", app_path,
            "--server.port", str(STREAMLIT_PORT),
            "--server.headless", "true",
            "--server.runOnSave", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print("Starting Streamlit", end="")
    for _ in range(20):
        time.sleep(1)
        print(".", end="", flush=True)
        if is_port_open(STREAMLIT_PORT):
            print(" ready!")
            break
    else:
        print("\n⚠️  Timed out — check that streamlit is installed in your environment.")

STREAMLIT_URL = f"http://localhost:{STREAMLIT_PORT}"
print(f"\nApp URL: {STREAMLIT_URL}")

In [ ]:
# Embed the running Streamlit app inline.
# Works in classic Jupyter Notebook and most JupyterLab configurations.
# In VS Code notebooks the IFrame may be blocked — use the link instead.
display(HTML(f"""
<div style="border:2px solid #e2e8f0; border-radius:8px; padding:12px;
            margin:8px 0; background:#f8fafc; font-family:sans-serif">
  <strong>❤️ Heart Disease Risk Predictor — Streamlit App</strong><br><br>
  <a href="{STREAMLIT_URL}" target="_blank" style="color:#1d4ed8">
    Open in a new tab → {STREAMLIT_URL}
  </a>
  <p style="color:#64748b; font-size:0.9em; margin-top:8px">
    Use the sidebar to navigate between Home, Prediction, Model Performance, and About.
  </p>
</div>
"""))

display(IFrame(src=STREAMLIT_URL, width="100%", height=700))

### Streamlit Model Performance page — replicated inline

The cells below reproduce the exact charts from the **Model Performance** page of the Streamlit app. They read from `models/training_results.json`, the same file the app reads, so the numbers are always in sync.

In [ ]:
# Load the saved results and display as a table (mirrors the Streamlit dataframe widget)
with RESULTS_PATH.open() as f:
    saved_results = json.load(f)

results_display = pd.DataFrame(saved_results).T.round(4)
results_display.columns = [c.replace("_", " ").title() for c in results_display.columns]
results_display.index.name = "Model"

print("=== Model Performance Table (Streamlit — Model Performance page) ===")
results_display

In [ ]:
# Row 1 of Streamlit charts: Accuracy and ROC AUC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["accuracy", "roc_auc"]):
    vals = {name: saved_results[name][metric] for name in saved_results}
    bars = ax.bar(vals.keys(), vals.values(),
                  color=["steelblue", "tomato", "seagreen"], edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, vals.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")

fig.suptitle("Streamlit — Model Performance page (row 1)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Row 2 of Streamlit charts: Precision, Recall, F1
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    vals = {name: saved_results[name][metric] for name in saved_results}
    bars = ax.bar(vals.keys(), vals.values(),
                  color=["steelblue", "tomato", "seagreen"], edgecolor="white", alpha=0.85)
    for bar, v in zip(bars, vals.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.set_title(metric.title())
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")

fig.suptitle("Streamlit — Model Performance page (row 2)", fontsize=11)
plt.tight_layout()
plt.show()

---
## 12. Ethical Reflection

*(Swedish as per course requirements)*

### Dataset

Datasetet kommer från UCI Heart Disease (Cleveland-subset) och innehåller 303 patientposter
med 13 kliniska egenskaper. Målvariabeln `target` anger om patienten har hjärtsjukdom (1) eller inte (0).

Majoriteten av patienter är medelålders män — en skevhet som kan påverka modellens prestanda
för underrepresenterade grupper såsom kvinnor och äldre patienter.

### Modellval

Fyra modeller tränades och jämförs: Logistic Regression (baseline), Random Forest (primärmodell),
Decision Tree (för att illustrera värdet av ensemblemetoder) och XGBoost (tillagd i EC5 —
industristandard för tabulärdata som bygger träd sekventiellt).

Random Forest valdes som primärmodell baserat på högst noggrannhet och ROC AUC.

### Hyperparametertuning

GridSearchCV med 5-faldig korsvalidering användes för att optimera alla tre basmodeller.
Decision Tree förbättrades mest (+8.2% accuracy). XGBoost tränas med standardparametrar
och förväntas förbättras ytterligare med tuning.

### SHAP — tolkbarhet

Random Forest och XGBoost är "black box"-modeller. SHAP-analys (Section 10) adresserar
detta direkt genom att förklara varje prediktion på feature-nivå. De tre mest inflytelserika
features är **thal**, **ca** och **cp**. En kliniker kan nu se exakt vilka faktorer som
drev en specifik högriskprediktion — vilket är ett krav i medicinsk beslutsstöd.

### Resultat

Random Forest uppnådde bäst resultat på samtliga mätetal. I ett medicinskt sammanhang är
**Recall** det mest kritiska måttet — ett missat fall av hjärtsjukdom (falskt negativt) är
allvarligare än ett falskt larm.

### Etiskt resonemang

Modellen förstärker potentiellt befintliga biaser om datasetet inte inkluderar tillräckligt
många underrepresenterade grupper. Innan modellen används kliniskt bör den valideras på ett
mer representativt dataset.

Valet att använda Cleveland-subsettet (303 rader) framför det kombinerade datasetet är
dokumenterat transparent — `ca` och `thal` är de två viktigaste features och kan inte
reliabelt imputeras för de övriga UCI-dataseten.

**Sammanfattning:** Modellen är ett pedagogiskt verktyg — inte ett medicinskt diagnostikinstrument.
Prediktioner ska alltid kompletteras med kliniskt omdöme och diagnostiska tester.


---
## 13. Terminal App

The project includes an interactive terminal app (`src/terminal_app.py`) built around the `HeartApp` class. It has four components:

| Component | Description |
|-----------|-------------|
| `FeatureInfo` | Dataclass holding metadata (name, description, data type) for each of the 13 clinical features |
| `__init__` | Loads the trained model from disk and defines all 13 features with descriptions |
| `prompt_for_inputs()` | Interactively prompts the user for each feature value, validates input type, and retries on invalid input |
| `predict()` | Takes a list of 13 values, runs inference and returns the binary prediction and probability |
| `run()` | Main loop: calls `prompt_for_inputs()`, then `predict()`, prints the result, and asks if the user wants to assess another patient |

The cells below demonstrate each component programmatically (without interactive input) so they can run in the notebook.

To run the full interactive version in a terminal:
```bash
py -m src.main --app
```


In [ ]:
# --- FeatureInfo and feature list ---
# Show all 13 features with their descriptions and expected data types.
# This is what prompt_for_inputs() uses to guide the user.
from src.terminal_app import HeartApp, FeatureInfo

app = HeartApp(str(MODEL_PATH))

print(f"{'Feature':<12} {'Type':<8} {'Description'}")
print("-" * 60)
for f in app.features:
    print(f"{f.name:<12} {f.data_type:<8} {f.description}")

In [ ]:
# --- predict() ---
# Demonstrate predict() with three example patients.
# In the real app these values come from prompt_for_inputs().

patients = {
    "Low risk patient": [45, 0, 1, 130, 250, 1, 0, 160, 1, 2.0, 2, 1, 3],
    "High risk patient": [65, 1, 0, 150, 300, 0, 2, 100, 0, 0.5, 1, 2, 7],
    "Borderline patient": [55, 1, 2, 120, 200, 0, 1, 140, 0, 1.0, 1, 0, 3],
}

print(f"{'Patient':<22} {'Result':<10} {'Confidence'}")
print("-" * 45)
for label, values in patients.items():
    prediction, probability = app.predict(values)
    risk = "LIKELY" if prediction == 1 else "UNLIKELY"
    print(f"{label:<22} {risk:<10} {probability:.1%}")

In [ ]:
# --- run() — full loop simulation ---
# Simulate the complete run() loop with mocked input
# to show the full user experience without blocking the notebook.
from unittest.mock import patch

# Simulate: one patient assessment, then exit
mock_inputs = [
    '55', '1', '0', '120', '200', '0', '1', '140', '0', '1.0', '1', '0', '3',  # feature values
    'n'  # exit
]

with patch('builtins.input', side_effect=mock_inputs):
    app.run()

---
## 14. CI/CD — GitHub Actions

ProjectEC5.1 uses **two separate workflows** for fast feedback and thorough PR gates:

- **`tests.yml`** — runs on every push. Full test suite + coverage check.
- **`pr_checks.yml`** — runs on every pull request. Everything in `tests.yml` plus flake8, bandit, mypy.

Both workflows use **Node.js 24** (`FORCE_JAVASCRIPT_ACTIONS_TO_NODE20: true` is replaced) ahead of GitHub's Node.js 20 deprecation deadline of 16 June 2026.

---

### `tests.yml` — push workflow

```yaml
# CI/CD Workflow — Heart Disease Prediction Project (EC5.1)
# Runs automatically on every push to any branch.
# Ensures no commit breaks the 66 tests or drops coverage below 84%.

name: Tests

on: [push]

env:
  FORCE_JAVASCRIPT_ACTIONS_TO_NODE24: true

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Prepare combined dataset
        run: py -m src.data_preparation

      - name: Test data preparation
        run: pytest tests/test_data_preparation.py -v

      - name: Test data processing
        run: pytest tests/test_data_processing.py -v

      - name: Test model training
        run: pytest tests/test_model_training.py -v

      - name: Test terminal app
        run: pytest tests/test_app.py -v

      - name: Test main CLI
        run: pytest tests/test_main.py -v

      - name: Coverage check
        run: pytest tests/ --cov=src --cov-report=term-missing --cov-report=html --cov-fail-under=84

      - name: Upload coverage report
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report
          path: htmlcov/
```

---

### `pr_checks.yml` — pull request workflow

```yaml
# PR Checks Workflow — Heart Disease Prediction Project (EC5.1)
# Runs on every pull request — everything in tests.yml plus code quality gates.
# All checks must pass before a branch can be merged to main.

name: PR Checks

on: [pull_request]

env:
  FORCE_JAVASCRIPT_ACTIONS_TO_NODE24: true

jobs:
  pr-checks:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Prepare combined dataset
        run: py -m src.data_preparation

      - name: Run full test suite with coverage
        run: pytest tests/ --cov=src --cov-report=term-missing --cov-report=html --cov-fail-under=84

      - name: Upload coverage report
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: pr-coverage-report
          path: htmlcov/

      - name: flake8 — code style check
        run: |
          pip install flake8
          flake8 src/ --max-line-length=100

      - name: bandit — security scan
        run: |
          pip install bandit
          bandit -r src/ -ll

      - name: mypy — type checking
        run: |
          pip install mypy
          mypy src/ --ignore-missing-imports
```

---

To run all checks locally:

```bash
# Tests and coverage
pytest tests/ -v
pytest tests/ --cov=src --cov-report=html   # generates htmlcov/index.html

# Code style
flake8 src/ --max-line-length=100

# Security scan
bandit -r src/ -ll

# Type check
mypy src/ --ignore-missing-imports
```


---
## 15. Summary

| Step | What happened |
|------|---------------|
| Environment check | Confirmed `.venv` kernel is active and all packages present |
| Data preparation | `data_preparation.py` loaded Cleveland data, imputed 6 missing values, saved `heart_combined.csv` |
| Data loading | UCI Heart Disease CSV loaded and cleaned via `DataProcessor` |
| EDA | Target balance, feature distributions, correlations visualised |
| Training | Logistic Regression, Random Forest and Decision Tree trained on 80% split |
| Saving | `.pkl` model files and `training_results.json` written to `models/` |
| Evaluation | Confusion matrices, ROC curves, feature importances, classification reports |
| Hyperparameter tuning | GridSearchCV, 5-fold CV — Decision Tree +8.2% accuracy — biggest winner |
| XGBoost | 4th model trained — matches LR baseline (0.8689) without tuning |
| SHAP | Beeswarm (RF + XGBoost) + waterfall — thal, ca, cp confirmed as top-3 features |
| Streamlit | App launched on port 8501, embedded inline, charts replicated in notebook |
| Ethical reflection | Dataset bias, SHAP explainability, clinical limitations documented |
| Terminal app | Interactive CLI demonstrated — `py -m src.main --app` |
| CI/CD | `tests.yml` (push) + `pr_checks.yml` (PR: flake8 + bandit + mypy), Node.js 24, 66 tests |

---

### The Evolving System — EC3 → EC4 → EC5

| Version | Goal | Key finding | Conclusion |
|---------|------|-------------|------------|
| **EC3** | Build baseline pipeline | RF 0.885 — looked strong | ⚠️ Later found inflated by 723 duplicate rows |
| **EC4** | Add Kaggle as new data source | More data degraded accuracy 0.885 → 0.643 | Data quality beats data quantity ✅ |
| **EC4** | Understand why more data hurt | `ca` + `thal` missing from Kaggle (~28% importance) | Cleveland only — all 13 features — wins |
| **EC4** | Tune models on validated dataset | Decision Tree +8.2% from GridSearchCV | Tuning recovers weak models |
| **EC5** | Add XGBoost on solid foundation | Matches LR baseline without tuning | Confirms dataset quality is good |
| **EC5** | Explain predictions (address ethics) | SHAP confirms `thal`, `ca`, `cp` as top features | EC4 feature decisions validated |

**EC3** built the pipeline. **EC4** proved that engineering rigour matters more than adding data —
investigating, deduplicating and justifying the dataset choice turned a flawed 0.885 into an
honest 0.902. **EC5** built on that solid foundation to add a competitive 4th model and
explainability that validates every previous decision.

Each version solved a real problem discovered in the previous one.

---

### Git workflow for EC5.1

```bash
# 1. Start from clean main
git checkout main
git pull origin main

# 2. Create feature branch
git checkout -b fix/ec5.1-notebook-cicd-sections

# 3. Stage all changes
git add notebooks/analysis.ipynb
git add .github/workflows/tests.yml
git add .github/workflows/pr_checks.yml
git add src/main.py
git add tests/test_main.py
git add tests/test_data_processing.py
git add report.md
git add README.md

# 4. Commit
git commit -m "fix: EC5.1 — section numbering, pr_checks.yml, Node.js 24, 66 tests, relative paths, evolution results"

# 5. Push and open PR — all pr_checks.yml gates must pass before merge
git push origin fix/ec5.1-notebook-cicd-sections

# 6. After merge — tag the release
git checkout main
git pull origin main
git tag -a v5.1 -m "ProjectEC5.1 — complete fix release with evolution results"
git push origin v5.1
```
